# Closure-Risk Analysis (synthetic demo)

**Pipeline:** Cox PH → RF / XGBoost → SHAP → SBRI.

> This notebook runs entirely on **synthetic data** generated by
> `src/make_synthetic.py`. No real administrative or personal data is used.
> The purpose is to demonstrate a reproducible analysis pipeline.

In [ ]:
# Setup
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure the synthetic dataset exists (generate it if missing)
os.makedirs("../data", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

data_path = "../data/synthetic_facilities.csv"
if not os.path.exists(data_path):
    sys.path.append("../src")
    from make_synthetic import make_synthetic
    make_synthetic().to_csv(data_path, index=False)

df = pd.read_csv(data_path)
print(f"rows: {len(df)}, event rate: {df['event'].mean():.3f}")
df.head()

## 1. Kaplan–Meier survival curves
Survival over time by facility type.

In [ ]:
from lifelines import KaplanMeierFitter

kmf = KaplanMeierFitter()
fig, ax = plt.subplots(figsize=(7, 5))
for t in sorted(df["facility_type"].unique()):
    m = df["facility_type"] == t
    kmf.fit(df.loc[m, "time"], df.loc[m, "event"], label=f"type {t}")
    kmf.plot_survival_function(ax=ax)
ax.set_title("Kaplan–Meier by facility type")
ax.set_xlabel("years"); ax.set_ylabel("survival probability")
plt.tight_layout()
plt.savefig("../figures/km_curves.png", dpi=150)
plt.show()

## 2. Cox proportional hazards
Interpretable hazard ratios for each covariate.

In [ ]:
from lifelines import CoxPHFitter

cph = CoxPHFitter()
cph.fit(df, duration_col="time", event_col="event")
cph.print_summary()

fig, ax = plt.subplots(figsize=(7, 4))
cph.plot(ax=ax)
ax.set_title("Cox hazard ratios")
plt.tight_layout()
plt.savefig("../figures/cox_hr.png", dpi=150)
plt.show()

## 3. Random Forest / XGBoost
Nonlinear closure prediction; compare ROC-AUC.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import xgboost as xgb

feat = ["tenure_yrs", "competitors_1km", "floor_area", "facility_type"]
Xtr, Xte, ytr, yte = train_test_split(
    df[feat], df["event"], test_size=0.25, random_state=0
)

rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xtr, ytr)
xg = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    eval_metric="logloss", random_state=0
).fit(Xtr, ytr)

print("RF  AUC:", round(roc_auc_score(yte, rf.predict_proba(Xte)[:, 1]), 4))
print("XGB AUC:", round(roc_auc_score(yte, xg.predict_proba(Xte)[:, 1]), 4))

## 4. SHAP — driver attribution
Which features drive predicted closure risk.

In [ ]:
import shap

explainer = shap.TreeExplainer(xg)
shap_values = explainer.shap_values(Xte)

shap.summary_plot(shap_values, Xte, show=False)
plt.tight_layout()
plt.savefig("../figures/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. SBRI — transparent composite risk index
A simple, auditable weighted score for operational ranking / early warning.

In [ ]:
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

sbri = (
    0.45 * minmax(df["competitors_1km"])
    + 0.35 * (1 - minmax(df["floor_area"]))
    + 0.20 * (1 - minmax(df["tenure_yrs"]))
)
df["SBRI"] = (100 * minmax(sbri)).round(1)

# Highest-risk facilities by SBRI
df[["facility_type", "tenure_yrs", "competitors_1km", "SBRI", "event"]] \
    .sort_values("SBRI", ascending=False).head(10)